<a href="https://colab.research.google.com/github/CPTR295/Learning-Langchain/blob/main/RAGFromScratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q chromadb openai langchain_openai


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 require

In [2]:
import chromadb #In memmory vector store
chroma_client = chromadb.Client()

Ingestion

In [3]:
tourism_collection = chroma_client.create_collection(name="tourism_collection")

In [4]:
tourism_collection.add(
    documents=[
        "Paestum, Greek Poseidonia, ancient city in southern Italy near the west coast, 22 miles (35 km) southeast of modern Salerno and 5 miles (8 km) south of the Sele (ancient Silarus) River. Paestum is noted for its splendidly preserved Greek temples.",
        "Poseidonia was probably founded about 600 BC by Greek colonists from Sybaris, along the Gulf of Taranto, and it had become a flourishing town by 540, judging from its temples. After many years’ resistance the city came under the domination of the Lucanians (an indigenous Italic people) sometime before 400 BC, after which its name was changed to Paestum. Alexander, the king of Epirus, defeated the Lucanians at Paestum about 332 BC, but the city remained Lucanian until 273, when it came under Roman rule and a Latin colony was founded there. The city supported Rome during the Second Punic War. The locality was still prosperous during the early years of the Roman Empire, but the gradual silting up of the mouth of the Silarus River eventually created a malarial swamp, and Paestum was finally deserted after being sacked by Muslim raiders in AD 871. The abandoned site’s remains were rediscovered in the 18th century.",
        "The ancient Greek part of Paestum consists of two sacred areas containing three Doric temples in a remarkable state of preservation. During the ensuing Roman period a typical forum and town layout grew up between the two ancient Greek sanctuaries. Of the three temples, the Temple of Athena (the so-called Temple of Ceres) and the Temple of Hera I (the so-called Basilica) date from the 6th century BC, while the Temple of Hera II (the so-called Temple of Neptune) was probably built about 460 BC and is the best preserved of the three. The Temple of Peace in the forum is a Corinthian-Doric building begun perhaps in the 2nd century BC. Traces of a Roman amphitheatre and other buildings, as well as intersecting main streets, have also been found. The circuit of the town walls, which are built of travertine blocks and are 15–20 feet (5–6 m) thick, is about 3 miles (5 km) in circumference. In July 1969 a farmer uncovered an ancient Lucanian tomb that contained Greek frescoes painted in the early classical style. Paestum’s archaeological museum contains these and other treasures from the site."
    ],
    metadatas=[
        {"source": "https://www.britannica.com/place/Paestum"},
        {"source": "https://www.britannica.com/place/Paestum"},
        {"source": "https://www.britannica.com/place/Paestum"}
    ],
    ids=["paestum-br-01", "paestum-br-02", "paestum-br-03"]
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:07<00:00, 10.6MiB/s]


Q&A

In [5]:
results = tourism_collection.query(
    query_texts=["How many Doric temples are in Paestum"],
    n_results=1
)
results

{'ids': [['paestum-br-03']],
 'embeddings': None,
 'documents': [['The ancient Greek part of Paestum consists of two sacred areas containing three Doric temples in a remarkable state of preservation. During the ensuing Roman period a typical forum and town layout grew up between the two ancient Greek sanctuaries. Of the three temples, the Temple of Athena (the so-called Temple of Ceres) and the Temple of Hera I (the so-called Basilica) date from the 6th century BC, while the Temple of Hera II (the so-called Temple of Neptune) was probably built about 460 BC and is the best preserved of the three. The Temple of Peace in the forum is a Corinthian-Doric building begun perhaps in the 2nd century BC. Traces of a Roman amphitheatre and other buildings, as well as intersecting main streets, have also been found. The circuit of the town walls, which are built of travertine blocks and are 15–20 feet (5–6 m) thick, is about 3 miles (5 km) in circumference. In July 1969 a farmer uncovered an anci

In [6]:
results = tourism_collection.query(
    query_texts=["How many Doric temples are in Paestum"],
    n_results=2
)
results

{'ids': [['paestum-br-03', 'paestum-br-01']],
 'embeddings': None,
 'documents': [['The ancient Greek part of Paestum consists of two sacred areas containing three Doric temples in a remarkable state of preservation. During the ensuing Roman period a typical forum and town layout grew up between the two ancient Greek sanctuaries. Of the three temples, the Temple of Athena (the so-called Temple of Ceres) and the Temple of Hera I (the so-called Basilica) date from the 6th century BC, while the Temple of Hera II (the so-called Temple of Neptune) was probably built about 460 BC and is the best preserved of the three. The Temple of Peace in the forum is a Corinthian-Doric building begun perhaps in the 2nd century BC. Traces of a Roman amphitheatre and other buildings, as well as intersecting main streets, have also been found. The circuit of the town walls, which are built of travertine blocks and are 15–20 feet (5–6 m) thick, is about 3 miles (5 km) in circumference. In July 1969 a farmer 

Basic RAG

In [7]:
from google.colab import userdata
api_key = userdata.get("OPENROUTER_API_KEY")

In [12]:
from openai import OpenAI
openai_client = OpenAI(
    #model="nvidia/nemotron-3-ultra-550b-a55b:free", - model is given during chat completion
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
)

In [13]:
def query_vector_database(question):
  results = tourism_collection.query(
      query_texts=[question],
      n_results=1
  )
  results_text = results['documents'][0][0]
  return results_text


In [14]:
result_text = query_vector_database("How many Doric temples are in Paestum")
result_text

'The ancient Greek part of Paestum consists of two sacred areas containing three Doric temples in a remarkable state of preservation. During the ensuing Roman period a typical forum and town layout grew up between the two ancient Greek sanctuaries. Of the three temples, the Temple of Athena (the so-called Temple of Ceres) and the Temple of Hera I (the so-called Basilica) date from the 6th century BC, while the Temple of Hera II (the so-called Temple of Neptune) was probably built about 460 BC and is the best preserved of the three. The Temple of Peace in the forum is a Corinthian-Doric building begun perhaps in the 2nd century BC. Traces of a Roman amphitheatre and other buildings, as well as intersecting main streets, have also been found. The circuit of the town walls, which are built of travertine blocks and are 15–20 feet (5–6 m) thick, is about 3 miles (5 km) in circumference. In July 1969 a farmer uncovered an ancient Lucanian tomb that contained Greek frescoes painted in the ear

Naive Prompt Implementation

In [15]:
def prompt_template(question,text):
  return f"Read the following text and answer this question: {question}. \nText: {text}"

In [16]:
def execute_llm_prompt(prompt_input):
  prompt_response = openai_client.chat.completions.create(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    messages=[
        {"role": "system", "content": "You are an assistant for question-answering tasks."},
        {"role": "user", "content":prompt_input}
    ]
  )
  return prompt_response

In [17]:
trick_question = "How many columns have the three temples got in total"
tq_result_text = query_vector_database(trick_question)
tq_prompt_input = prompt_template(trick_question,tq_result_text)
tq_prompt_response = execute_llm_prompt(tq_prompt_input)
tq_prompt_response

ChatCompletion(id='gen-1794307968-Rzl0YOIyaGpFMfmNhiht', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The text does not provide any information about the number of columns in the three temples. It mentions the three Doric temples (Temple of Athena/Ceres, Temple of Hera I, and Temple of Hera II/Neptune) and their dates of construction, but it does not state how many columns each temple has or a total column count.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user is asking about how many columns the three temples have in total. I need to read the text carefully to see if it mentions the number of columns for each temple or a total.\n\nLooking at the text:\n- It mentions three Doric temples: Temple of Athena (Temple of Ceres), Temple of Hera I, and Temple of Hera II (Temple of Neptune)\n- It gives dates for each temple\n- It mentions the Temple of Peace is a Cori

In [19]:
def prompt_template(question, text): #With guardrails
    return f'Use the following pieces of retrieved context to answer the question. Only use the retrieved context to answer the question. If you don\'t know the answer, or the answer is not contained in the retrieved context, just say that you don\'t know. Use three sentences maximum and keep the answer concise. \nQuestion: {question}\nContext: {text}. Remember: if you do not know, just say: I do not know. Do not make up an answer. For example do not say the three temples have got a total of three columns. \nAnswer:'

In [20]:
trick_question = "How many columns have the three temples got in total"
tq_result_text = query_vector_database(trick_question)
tq_prompt_input = prompt_template(trick_question,tq_result_text)
tq_prompt_response = execute_llm_prompt(tq_prompt_input)
tq_prompt_response

ChatCompletion(id='gen-1794308043-icEvv3SKHXt3G9qs0VoL', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I do not know.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning="The user is asking about the total number of columns across three temples, but the provided context doesn't mention the number of columns for any of the temples. The context describes three Doric temples (Temple of Athena, Temple of Hera I, Temple of Hera II) and their dates, preservation states, and alternative names, but gives no information about column counts. Since the answer isn't in the retrieved context, I must say I don't know.", reasoning_details=[{'type': 'reasoning.text', 'text': "The user is asking about the total number of columns across three temples, but the provided context doesn't mention the number of columns for any of the temples. The context describes three Doric temples (Temple of At

My Chatbot

In [21]:
def my_chatbot(question):
  results_text = query_vector_database(question)  #Retrieve the content from vector store
  prompt_input = prompt_template(question,results_text) #Create the prompe
  prompt_response = execute_llm_prompt(prompt_input) #Execute LLM
  return prompt_response

In [23]:
question = """Let me know how many temples there are in Paestum, who constructed them, and what architectural style they are"""
result = my_chatbot(question)
print(result)

ChatCompletion(id='gen-1794308214-spqFMKoszudlYuSJ4PF3', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='There are three Doric temples in the ancient Greek part of [ADDRESS]: the Temple of Athena (Temple of Ceres), the Temple of Hera I, and the Temple of Hera II (Temple of Neptune), dating from the 6th century BC and circa 460 BC. The Temple of Peace in the forum is a Corinthian-Doric building from the 2nd century BC. The retrieved context does not specify who constructed these temples.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user is asking about temples in [ADDRESS], who constructed them, and their architectural style. The context mentions three Doric temples in the ancient Greek part of [ADDRESS]: Temple of Athena (Temple of Ceres), Temple of Hera I, and Temple of Hera II (Temple of Neptune). They date from 6th century BC and 460 BC. The context doesn\'t ex